# 🇲🇦 Vasco LLM — entraînement (Kaggle, GPU gratuit)

Fine-tuning **LoRA (QLoRA 4-bit)** de **Qwen3 4B Instruct** (licence Apache 2.0) sur les données de Vasco.

**Avant de lancer :**
1. Sur votre PC : `python vasco-llm/prepare_data.py` → crée `vasco-llm/build/train.jsonl` et `eval.jsonl`.
2. Kaggle → *Datasets* → *New Dataset* → envoyez les 2 fichiers du dossier `build/`, nommez-le **vasco-data**.
3. Ici : *Add Input* → votre dataset **vasco-data** ; *Settings* → **Accelerator : GPU T4 x2** (ou P100) ; **Internet : On**.
4. *Run All*. Durée : ~20 min à 2 h selon la quantité de données.
5. À la fin, téléchargez `vasco-gguf/…Q4_K_M.gguf` (onglet *Output*) puis sur votre PC :
   `python vasco-llm/create_ollama_model.py --gguf <fichier.gguf> --name mr1:v1`

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"  # Qwen3 4B Instruct, Apache 2.0
import glob, os
_found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
DATA_DIR = os.path.dirname(_found[0]) if _found else "/kaggle/input/vasco-data"  # dossier de train.jsonl / eval.jsonl
print("Données :", DATA_DIR)
MAX_LEN = 2048
EPOCHS = 2            # 2-3 pour quelques centaines d'exemples ; 1 au-delà de 20 000
LEARNING_RATE = 2e-4
LORA_RANK = 16

In [ ]:
import os, json, torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_RANK * 2, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

files = {"train": f"{DATA_DIR}/train.jsonl"}
eval_path = f"{DATA_DIR}/eval.jsonl"
if os.path.exists(eval_path) and os.path.getsize(eval_path) > 0:
    files["eval"] = eval_path
ds = load_dataset("json", data_files=files)

with open(f"{DATA_DIR}/train.jsonl", encoding="utf-8") as f:
    PERSONA = json.loads(f.readline())["messages"][0]["content"]  # la personnalité de Vasco

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}

ds = ds.map(to_text, batched=True, remove_columns=["messages"])
print(ds)
print(ds["train"][0]["text"][:800])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

has_eval = "eval" in ds
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["eval"] if has_eval else None,
    args=SFTConfig(
        output_dir="outputs",
        dataset_text_field="text",
        packing=False,  # packing (on by default in recent Unsloth) broke the response-only masking
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="epoch" if has_eval else "no",
        save_strategy="no",
        report_to="none",
        seed=42,
    ),
)
# Le modèle n'apprend que les réponses de Vasco (pas à répéter les questions).
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
)
# Check: only Vasco's answers are trained on (masked labels = -100). If this shows the system prompt
# or the question, or tool tags, the masking is wrong: stop here instead of wasting an hour of GPU.
row = trainer.train_dataset[0]
trained = [t for t, l in zip(row["input_ids"], row["labels"]) if l != -100]
trained_text = tokenizer.decode(trained)
print("Tokens appris (début) :", trained_text[:400])
assert trained, "Aucun token appris : marqueurs de réponse introuvables"
assert "<|im_start|>user" not in trained_text and "tool_call" not in trained_text, "Masquage incorrect"
stats = trainer.train()
print(stats)

In [ ]:
# Petit test avant export
FastLanguageModel.for_inference(model)
BROKEN = []
for q in ["salam, chkoun nta?", "شنو هي أحسن طريقة باش نتعلم البرمجة؟", "Bonjour, tu parles darija ?"]:
    msgs = [{"role": "system", "content": PERSONA}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.6, top_p=0.9, do_sample=True)
    answer = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    print("❓", q, "\n🤖", answer, "\n")
    if not answer.strip() or "tool_call" in answer:
        BROKEN.append(q)
# A broken model must not be exported (and installed in Ollama) as if it were fine.
if BROKEN:
    raise RuntimeError(f"Réponses cassées pour {BROKEN} : export annulé")

In [ ]:
# Export : adaptateur LoRA (petit) + modèle complet GGUF Q4_K_M pour Ollama (~2,5 Go)
model.save_pretrained("vasco-lora")
tokenizer.save_pretrained("vasco-lora")
model.save_pretrained_gguf("vasco-gguf", tokenizer, quantization_method="q4_k_m")
!ls -la vasco-gguf